In [1]:
import time
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("phase3-model-training").getOrCreate()
tx = spark.read.parquet("/home/jovyan/work/data/processed/train_features")
total_rows = tx.count()
print(f"features table: {total_rows:,} rows × {len(tx.columns)} cols")

features table: 590,540 rows × 298 cols


In [2]:
max_day = tx.agg(F.max("day")).collect()[0][0]
cutoff  = max_day - 17                      # last 18 days = unseen future
train = tx.filter(F.col("day") <= cutoff)
test  = tx.filter(F.col("day") >  cutoff)

for name, df in [("train", train), ("test", test)]:
    n   = df.count()
    fr  = df.filter(F.col("isFraud") == 1).count() / n * 100
    print(f"{name}: {n:,} rows | fraud rate {fr:.2f}%")
print(f"split: train days ≤ {cutoff}, test days {cutoff+1}–{max_day}")

train: 543,948 rows | fraud rate 3.47%
test: 46,592 rows | fraud rate 3.88%
split: train days ≤ 165, test days 166–182


In [3]:
num_cols = [c for c, t in train.dtypes
            if t in ("int", "bigint", "double", "float")
            and c not in ("isFraud", "TransactionID")]

t0 = time.time()
meds = train.approxQuantile(num_cols, [0.5], 0.001)

med_map = {}
for c, m in zip(num_cols, meds):
    if not m or m[0] is None:
        continue
    t = next(t for cc, t in train.dtypes if cc == c)
    v = int(round(m[0])) if t in ("int", "bigint") else float(m[0])
    med_map[c] = v

train = train.na.fill(med_map)
test  = test.na.fill(med_map)
print(f"{len(med_map)} numeric cols median-imputed in {time.time()-t0:.1f}s")

282 numeric cols median-imputed in 89.5s


In [4]:
from pyspark.ml.feature import StringIndexer

str_cols = [c for c, t in train.dtypes if t == "string"]
idx_cols = [c + "_idx" for c in str_cols]

# idempotence prep: wipe any _idx columns left by an earlier attempt
train = train.drop(*idx_cols)
test  = test.drop(*idx_cols)

train = train.na.fill("__NA__", subset=str_cols)
test  = test.na.fill("__NA__", subset=str_cols)

t0 = time.time()
for c in str_cols:
    indexer = StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    model   = indexer.fit(train)        # estimator  -> model (fit on TRAIN only)
    train   = model.transform(train)    # model      -> transformed columns
    test    = model.transform(test)     # train-fitted model applied to test
    train   = train.drop(c)
    test    = test.drop(c)

print(f"indexed {len(str_cols)} string cols in {time.time()-t0:.1f}s")
print(f"train now: {len(train.columns)} cols | test now: {len(test.columns)} cols")

indexed 14 string cols in 25.6s
train now: 298 cols | test now: 298 cols


In [5]:
t0 = time.time()
n_train = train.cache().count()          # count ONCE; cache so reuses are cheap
fraud_n = train.filter(F.col("isFraud") == 1).count()
legit_n = n_train - fraud_n
w_fraud = n_train / (2 * fraud_n)
w_legit = n_train / (2 * legit_n)
print(f"train: {n_train:,} rows | fraud {fraud_n:,} / legit {legit_n:,}")
print(f"class weights -> fraud: {w_fraud:.2f}x, legit: {w_legit:.2f}x")

train = train.withColumn("weight", F.when(F.col("isFraud") == 1, w_fraud).otherwise(w_legit))
test  = test.withColumn("weight", F.when(F.col("isFraud") == 1, w_fraud).otherwise(w_legit))

# strip nominal metadata from _idx cols: trees would otherwise demand
# maxBins >= category count (your feature 285 = 62 values, DeviceInfo-family worse)
idx_cols = [c for c in train.columns if c.endswith("_idx")]
print(f"treating {len(idx_cols)} _idx cols as numeric (frequency encoding)")
for c in idx_cols:
    train = train.withColumn(c, F.col(c) * 1.0)
    test  = test.withColumn(c, F.col(c) * 1.0)

# TransactionDT excluded: same info as `day` but with runaway magnitude;
# `day` (relative, bounded) carries it safely
assembler_inputs = [c for c, t in train.dtypes
                    if t in ("int", "bigint", "double", "float")
                    and c not in ("isFraud", "TransactionID", "weight", "TransactionDT")]

from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="keep")

train_vec = assembler.transform(train).select("TransactionID", "day", "isFraud", "features", "weight")
test_vec  = assembler.transform(test).select("TransactionID", "day", "isFraud", "features", "weight").cache()
print(f"test_vec cached: {test_vec.count():,} rows | {len(assembler_inputs)} features")

train_s = train_vec.sample(False, 0.05, seed=42).cache()
print(f"5% tuning sample: {train_s.count():,} rows | done in {time.time()-t0:.1f}s")

train: 543,948 rows | fraud 18,854 / legit 525,094
class weights -> fraud: 14.43x, legit: 0.52x
treating 14 _idx cols as numeric (frequency encoding)
test_vec cached: 46,592 rows | 295 features
5% tuning sample: 27,365 rows | done in 99.8s


In [6]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

def score(pred):
    ev = BinaryClassificationEvaluator(labelCol="isFraud", rawPredictionCol="probability")
    tp = pred.filter("isFraud=1 and prediction=1").count()
    fp = pred.filter("isFraud=0 and prediction=1").count()
    fn = pred.filter("isFraud=1 and prediction=0").count()
    tn = pred.filter("isFraud=0 and prediction=0").count()
    prec = tp/(tp+fp) if tp+fp else 0.0
    rec  = tp/(tp+fn) if tp+fn else 0.0
    f1   = 2*prec*rec/(prec+rec) if prec+rec else 0.0
    return dict(prAUC=round(ev.setMetricName("areaUnderPR").evaluate(pred), 4),
                rocAUC=round(ev.setMetricName("areaUnderROC").evaluate(pred), 4),
                precision=round(prec, 3), recall=round(rec, 3),
                f1=round(f1, 3), accuracy=round((tp+tn)/(tp+fp+fn+tn), 4))

def show_scores(name, sco, elapsed):
    label = max(len(name) + 4, 28)
    print(f"=== {name} | fit+score {elapsed:.1f}s " + "=" * max(0, label - len(name)))
    print(f"{'metric':<12}{'train':>10}{'test':>10}")
    for k in ["prAUC", "rocAUC", "precision", "recall", "f1", "accuracy"]:
        print(f"{k:<12}{sco['train'][k]:>10}{sco['test'][k]:>10}")

print("scorer ready")

scorer ready


In [7]:
# Model 1: Logistic Regression
from pyspark.ml.classification import LogisticRegression

t0 = time.time()
lr = LogisticRegression(labelCol="isFraud", featuresCol="features",
                        weightCol="weight", maxIter=20, regParam=0.01)
lr_m   = lr.fit(train_s)
lr_sco = {"train": score(lr_m.transform(train_s)), "test": score(lr_m.transform(test_vec))}
show_scores("Logistic Regression", lr_sco, time.time() - t0)

=== Logistic Regression | fit+score 29.9s =========
metric           train      test
prAUC           0.3705    0.1579
rocAUC           0.866    0.8252
precision        0.129     0.114
recall           0.734     0.738
f1               0.219     0.197
accuracy        0.8243    0.7665


In [8]:
# Model 2: Random Forest
from pyspark.ml.classification import RandomForestClassifier

t0 = time.time()
rf = RandomForestClassifier(labelCol="isFraud", featuresCol="features",
                            weightCol="weight", numTrees=60, maxDepth=8, seed=42)
rf_m   = rf.fit(train_s)
rf_sco = {"train": score(rf_m.transform(train_s)), "test": score(rf_m.transform(test_vec))}
show_scores("Random Forest", rf_sco, time.time() - t0)

=== Random Forest | fit+score 38.8s ===============
metric           train      test
prAUC           0.5948    0.4336
rocAUC          0.9253    0.8633
precision        0.245     0.237
recall           0.762     0.604
f1               0.371      0.34
accuracy        0.9132    0.9091


In [9]:
# Model 3: Gradient-Boosted Trees 
from pyspark.ml.classification import GBTClassifier

t0 = time.time()
gbt = GBTClassifier(labelCol="isFraud", featuresCol="features",
                    weightCol="weight", maxIter=50, maxDepth=5, stepSize=0.1, seed=42)
gbt_m   = gbt.fit(train_s)
gbt_sco = {"train": score(gbt_m.transform(train_s)), "test": score(gbt_m.transform(test_vec))}
show_scores("Gradient-Boosted Trees", gbt_sco, time.time() - t0)

=== Gradient-Boosted Trees | fit+score 98.3s ======
metric           train      test
prAUC           0.7133    0.4476
rocAUC          0.9726    0.8587
precision        0.263     0.216
recall           0.907      0.65
f1               0.407     0.325
accuracy        0.9112    0.8951


In [10]:
rows = []
for name, sco in [("LR", lr_sco), ("RF", rf_sco), ("GBT", gbt_sco)]:
    for split in ("train", "test"):
        row = {"model": name, "split": split}
        row.update(sco[split])
        rows.append(row)

comp = pd.DataFrame(rows)
print(comp.to_string(index=False))

best = comp[comp["split"] == "test"].sort_values("prAUC", ascending=False).iloc[0]
print(f"\nwinner on test prAUC: {best['model']} at {best['prAUC']:.4f}")

model split  prAUC  rocAUC  precision  recall    f1  accuracy
   LR train 0.3705  0.8660      0.129   0.734 0.219    0.8243
   LR  test 0.1579  0.8252      0.114   0.738 0.197    0.7665
   RF train 0.5948  0.9253      0.245   0.762 0.371    0.9132
   RF  test 0.4336  0.8633      0.237   0.604 0.340    0.9091
  GBT train 0.7133  0.9726      0.263   0.907 0.407    0.9112
  GBT  test 0.4476  0.8587      0.216   0.650 0.325    0.8951

winner on test prAUC: GBT at 0.4476


In [11]:
fi = pd.DataFrame({"feature": assembler_inputs,
                   "importance": gbt_m.featureImportances.toArray()})
fi = fi.sort_values("importance", ascending=False).reset_index(drop=True)
fi["cum_pct"] = fi["importance"].cumsum() * 100
top = fi.head(15).copy()
top["share_pct"] = top["importance"] * 100
print(top[["feature", "importance", "share_pct", "cum_pct"]].round(4).to_string(index=False))

velocity_hits = [f for f in top["feature"]
                 if f.startswith("card1_") or f.startswith("amt_vs")]
print(f"\nengineered velocity features in top-15: {velocity_hits if velocity_hits else 'none'}")

       feature  importance  share_pct  cum_pct
            C8      0.0570     5.7026   5.7026
 card1_amt_avg      0.0387     3.8693   9.5719
         card2      0.0371     3.7070  13.2789
           day      0.0336     3.3645  16.6434
           C13      0.0319     3.1855  19.8289
          V258      0.0313     3.1297  22.9586
TransactionAmt      0.0295     2.9473  25.9059
 card1_amt_24h      0.0277     2.7654  28.6712
            D3      0.0244     2.4398  31.1110
            C1      0.0243     2.4263  33.5373
   amt_vs_card      0.0234     2.3404  35.8777
           D11      0.0225     2.2454  38.1231
           C14      0.0219     2.1856  40.3087
           V70      0.0210     2.1003  42.4090
         dist1      0.0204     2.0438  44.4528

engineered velocity features in top-15: ['card1_amt_avg', 'card1_amt_24h', 'amt_vs_card']


In [12]:
from pyspark.ml.classification import GBTClassifier
import numpy as np

n_pos = train_s.filter(F.col("isFraud") == 1).count()
n_neg = train_s.filter(F.col("isFraud") == 0).count()
ratio = n_neg / n_pos
print(f"tuning sample: {n_pos:,} fraud / {n_neg:,} legit | imbalance ratio {ratio:.1f}x")

train_s = train_s.drop("weight")     # rebuild per scheme; features vector untouched

schemes = {
    "balanced (28x)": (ratio,               1.0),
    "sqrt (~5x)":     (float(np.sqrt(ratio)), 1.0),
    "none (1x)":      (1.0,                 1.0),
}

shoot = []
for name, (w_pos, w_neg) in schemes.items():
    t0 = time.time()
    wtrain = train_s.withColumn("w", F.when(F.col("isFraud") == 1, w_pos).otherwise(w_neg))
    model  = GBTClassifier(labelCol="isFraud", featuresCol="features", weightCol="w",
                           maxIter=50, maxDepth=5, stepSize=0.1, seed=42).fit(wtrain)
    tr  = score(model.transform(wtrain))
    te  = score(model.transform(test_vec))
    shoot.append({"scheme": name, "prAUC_train": tr["prAUC"], "prAUC_test": te["prAUC"],
                  "rocAUC_test": te["rocAUC"], "gap": round(tr["prAUC"] - te["prAUC"], 4),
                  "secs": round(time.time() - t0, 1)})
    print(f"done {name:>16} in {time.time()-t0:.1f}s")

shoot_df = pd.DataFrame(shoot)
print(shoot_df.to_string(index=False))

tuning sample: 920 fraud / 26,445 legit | imbalance ratio 28.7x
done   balanced (28x) in 131.5s
done       sqrt (~5x) in 101.8s
done        none (1x) in 113.9s
        scheme  prAUC_train  prAUC_test  rocAUC_test    gap  secs
balanced (28x)       0.7134      0.4532       0.8585 0.2602 131.5
    sqrt (~5x)       0.7240      0.4540       0.8747 0.2700 101.8
     none (1x)       0.7094      0.4538       0.8667 0.2556 113.9


In [13]:
best_scheme = shoot_df.sort_values("prAUC_test", ascending=False).iloc[0]["scheme"]
w_pos, w_neg = schemes[best_scheme]
print(f"continuing with weight scheme: {best_scheme}")

wtrain = train_s.withColumn("w", F.when(F.col("isFraud") == 1, w_pos).otherwise(w_neg))

configs = [
    {"name": "base            d5 i50  s0.1",      "maxDepth": 5, "maxIter": 50,  "stepSize": 0.1,  "subsamplingRate": 1.0},
    {"name": "shallow         d3 i50  s0.1",      "maxDepth": 3, "maxIter": 50,  "stepSize": 0.1,  "subsamplingRate": 1.0},
    {"name": "slow+deep       d3 i100 s0.05",     "maxDepth": 3, "maxIter": 100, "stepSize": 0.05, "subsamplingRate": 1.0},
    {"name": "row-subsampling d5 i50  s0.1 ss.8", "maxDepth": 5, "maxIter": 50,  "stepSize": 0.1,  "subsamplingRate": 0.8},
]

ctrl = []
for cfg in configs:
    t0 = time.time()
    model = GBTClassifier(labelCol="isFraud", featuresCol="features", weightCol="w",
                          seed=42, **{k: v for k, v in cfg.items() if k != "name"})
    m_fit = model.fit(wtrain)
    tr = score(m_fit.transform(wtrain))
    te = score(m_fit.transform(test_vec))
    ctrl.append({"config": cfg["name"],
                 "prAUC_train": tr["prAUC"], "prAUC_test": te["prAUC"],
                 "rocAUC_test": te["rocAUC"], "gap": round(tr["prAUC"] - te["prAUC"], 4),
                 "secs": round(time.time() - t0, 1)})
    print(f"done {cfg['name']} in {time.time()-t0:.1f}s")

ctrl_df = pd.DataFrame(ctrl)
print(ctrl_df.to_string(index=False))

continuing with weight scheme: sqrt (~5x)


done base            d5 i50  s0.1 in 109.2s
done shallow         d3 i50  s0.1 in 65.0s
done slow+deep       d3 i100 s0.05 in 132.1s
done row-subsampling d5 i50  s0.1 ss.8 in 107.6s
                           config  prAUC_train  prAUC_test  rocAUC_test    gap  secs
     base            d5 i50  s0.1       0.7240      0.4539       0.8747 0.2701 109.2
     shallow         d3 i50  s0.1       0.5336      0.4468       0.8695 0.0868  65.0
    slow+deep       d3 i100 s0.05       0.5353      0.4413       0.8677 0.0940 132.1
row-subsampling d5 i50  s0.1 ss.8       0.7228      0.4413       0.8766 0.2815 107.6


In [14]:
best = ctrl_df.sort_values("prAUC_test", ascending=False).iloc[0]
print("GBT recipe locked for Phase 4 (full-scale training):")
print(f"  weight scheme : {best_scheme}")
print(f"  config        : {best['config'].strip()}")
print(f"  test prAUC    : {best['prAUC_test']}  |  train-test gap: {best['gap']}")

GBT recipe locked for Phase 4 (full-scale training):
  weight scheme : sqrt (~5x)
  config        : base            d5 i50  s0.1
  test prAUC    : 0.4539  |  train-test gap: 0.2701


In [15]:
import sys, pyspark
print(sys.executable)        # expect /opt/conda/bin/python
print(pyspark.__version__)   # expect 3.5.0

/opt/conda/bin/python
3.5.0
